# Spin Lattice in Cirq

Due to its improved performance, here we implement the spin lattice RC in Cirq

### Imports

In [ ]:
# Linalg support
import numpy as np

# Cirq library
import cirq
from cirq.ops import PauliSum, PauliString
from cirq import Simulator

### Data Loading

### System Construction

In [ ]:
def create_ising_model_circuit(L, J, g, B, t):
    """Create a Cirq circuit for the time-dependent 2D Ising model."""
    circuit = cirq.Circuit()

    qubits = [cirq.GridQubit(i, j) for i in range(L) for j in range(L)]

    # Add the time-dependent magnetic field term
    for i in range(L):
        for j in range(L):
            circuit.append(cirq.Z(qubits[i * L + j]) ** B(t, i, j))

    # Add the Ising interaction term
    for i in range(L):
        for j in range(L):
            if i + 1 < L:
                circuit.append(cirq.ZZ(qubits[i * L + j], qubits[(i + 1) * L + j]) ** J)
            if j + 1 < L:
                circuit.append(cirq.ZZ(qubits[i * L + j], qubits[i * L + j + 1]) ** J)

    # Add the transverse field term
    for i in range(L):
        for j in range(L):
            circuit.append(cirq.X(qubits[i * L + j]) ** g)

    return circuit

In [ ]:
def time_evolution_cirq(L, J, g, B, total_time, num_time_steps):
    """
    Perform time evolution in Cirq.
    """
    qubits = [cirq.GridQubit(i, j) for i in range(L) for j in range(L)]

    simulator = Simulator()

    # Set up time steps
    times = np.linspace(0, total_time, num_time_steps)

    # Initialize the state
    initial_state = simulator.simulate(create_ising_model_circuit(L, J, g, B, 0)).final_state_vector

    # Perform time evolution
    states = []
    for t in times:
        circuit = create_ising_model_circuit(L, J, g, B, t)
        result = simulator.simulate(circuit, initial_state=initial_state)
        initial_state = result.final_state_vector
        states.append(result.final_state_vector)

    return states

### Simulation

In [ ]:
# Example: Time evolution of a 3x3 Ising lattice with a sinusoidal magnetic field
L = 3
J = 1.0
g = 0.5

# Define a time-dependent magnetic field function (e.g., a sinusoidal function)
def time_dependent_magnetic_field(t, i, j):
    return 0.5 * (1 + np.sin(2 * np.pi * t))

total_time = 1.0
num_time_steps = 100

# Perform time evolution
evolution_states = time_evolution_cirq(L, J, g, time_dependent_magnetic_field, total_time, num_time_steps)


### Measurements

In [ ]:
dm = cirq.density_matrix_from_state_vector(evolution_states[0])

In [ ]:
measurement_system = cirq.partial_trace_of_state_vector_as_mixture(evolution_states[0], keep_indices=[1, 2, 3])

In [ ]:
random_hermitian = cirq.testing.random_density_matrix(8)

In [ ]:
np.real(np.trace(cirq.density_matrix_from_state_vector(measurement_system[0][1]) @ random_hermitian))

### Readout Fitting

### Model Analysis